# Swiss Data Intelligence — Data Preparation

## Objective

This notebook prepares the analytical dataset for the Swiss Data Intelligence
machine-learning use case.

The objective is to investigate whether demographic characteristics and household
composition can help explain and predict differences in home-ownership rates across
Swiss cantons.

The analytical unit is **canton-year**, covering:

- 26 Swiss cantons
- 6 years (2019–2024)
- 156 canton-year observations

Three datasets from the Swiss Federal Statistical Office (BFS/FSO) are integrated:

1. Home ownership rate — target variable
2. Private households by household size — household features
3. Permanent resident population by sex, marital status and age — demographic features

National aggregates and non-cantonal observations are excluded from the analytical dataset.

In [1]:
# Standard library imports
import sys
from pathlib import Path

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
Path.cwd()

WindowsPath('G:/Otros ordenadores/Mi MacBook\xa0Pro/HSLU-CAS_ML-CAS Project/swiss-data-intelligence/notebooks')

In [3]:
# Resolve the project root from the notebook directory.
# The notebook is located in /notebooks, so the repository root is one level above.
PROJECT_ROOT = Path("..").resolve()

# Add the project root to the Python path so that reusable modules
# from the src package can be imported from this notebook.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import reusable project components after the project root
# has been made available to Python.
from src.data.ingestion import load_dataset

print(f"Project root: {PROJECT_ROOT}")

Project root: G:\Otros ordenadores\Mi MacBook Pro\HSLU-CAS_ML-CAS Project\swiss-data-intelligence


In [4]:
# Define the location of the raw BFS datasets used in the MVP.
# Keeping the paths centralized makes the notebook easier to maintain
# and avoids hardcoding absolute machine-specific paths.
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "bfs"

HOME_OWNERSHIP_PATH = (
    DATA_RAW
    / "housing"
    / "home-ownership-rate-by-canton"
    / "BFS_HOME_OWNERSHIP_RATE_BY_CANTON-2019_2024.csv"
)

HOUSEHOLDS_PATH = (
    DATA_RAW
    / "housing"
    / "private-households-by-household-size"
    / "PRIVATE_HOUSEHOLDS_BY_HOUSEHOLD_SIZE_2019_2024.csv"
)

POPULATION_PATH = (
    DATA_RAW
    / "demographics"
    / "population-by-canton-sex-marital-status-age"
    / "POPULATION_BY_CANTON_SEX_MARITAL_STATUS_AGE_2019_2024.csv"
)

In [5]:
# Verify that all raw datasets required by the MVP are available before
# starting any transformations. This provides an early and explicit failure
# point if the project is executed in a different environment.
datasets = {
    "Home ownership": HOME_OWNERSHIP_PATH,
    "Households": HOUSEHOLDS_PATH,
    "Population": POPULATION_PATH,
}

for name, path in datasets.items():
    print(f"{name:15} | exists={path.exists()} | {path.name}")

Home ownership  | exists=True | BFS_HOME_OWNERSHIP_RATE_BY_CANTON-2019_2024.csv
Households      | exists=True | PRIVATE_HOUSEHOLDS_BY_HOUSEHOLD_SIZE_2019_2024.csv
Population      | exists=True | POPULATION_BY_CANTON_SEX_MARITAL_STATUS_AGE_2019_2024.csv


## 2. Target Preparation — Home Ownership Rate

The home-ownership rate is the target variable of the supervised machine-learning problem.

The raw BFS dataset contains annual home-ownership rates for Switzerland and the 26 cantons from 2019 to 2024.

Since the analytical unit of this project is *canton-year*, the national Swiss aggregate is excluded. The resulting target dataset should therefore contain:

**26 cantons × 6 years = 156 observations**

Before transforming the data, the source is inspected and validated to confirm its expected temporal coverage, geographic coverage, and data quality.

In [6]:
# Load the home-ownership dataset using the reusable ingestion component
# developed during the ingestion phase of the project.
#
# From this point onwards, raw datasets should be loaded through
# load_dataset() rather than directly through pandas whenever possible.
home_ownership_raw = load_dataset(HOME_OWNERSHIP_PATH)

# Display a sample to verify that the dataset was parsed correctly.
home_ownership_raw.head()

,"﻿""TIME_PERIOD""",GEO,OBS_VALUE,OBS_CONFIDENCE,OBS_STATUS
0,2019,CH040,27.7,0.5,A
1,2019,CH021,38.8,0.6,A
2,2019,CH061,34.4,0.7,A
3,2019,CH062,42.9,3.5,A
4,2019,CH063,39.4,1.6,A


In [7]:
# Inspect the exact column names returned by the ingestion layer.
# repr() makes hidden characters and quotation marks visible.
for column in home_ownership_raw.columns:
    print(repr(column))

'\ufeff"TIME_PERIOD"'
'GEO'
'OBS_VALUE'
'OBS_CONFIDENCE'
'OBS_STATUS'


In [8]:
# Standardize column names before applying dataset-specific transformations.
# The BFS CSV contains a UTF-8 BOM and quotation marks in the first header.
# These characters are removed to obtain consistent column names.
home_ownership_raw.columns = (
    home_ownership_raw.columns
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
    .str.strip('"')
)

# Verify the normalized column names.
print(home_ownership_raw.columns.tolist())

['TIME_PERIOD', 'GEO', 'OBS_VALUE', 'OBS_CONFIDENCE', 'OBS_STATUS']


In [9]:
# Inspect the raw target dataset after standardizing the column names.
# This confirms its dimensions, temporal coverage, geographic coverage
# and the basic quality of the target variable.

print(f"Shape: {home_ownership_raw.shape}")
print(f"Years: {sorted(home_ownership_raw['TIME_PERIOD'].unique())}")
print(f"Number of geographies: {home_ownership_raw['GEO'].nunique()}")
print(f"Missing target values: {home_ownership_raw['OBS_VALUE'].isna().sum()}")

print("\nObservation status:")
print(home_ownership_raw["OBS_STATUS"].value_counts(dropna=False))

Shape: (162, 5)
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Number of geographies: 27
Missing target values: 0

Observation status:
OBS_STATUS
A    162
Name: count, dtype: int64


In [10]:
# Validate the assumptions identified during the Data Audit.
# These checks make the preparation pipeline fail explicitly if the
# structure or quality of the raw target dataset changes in the future.

EXPECTED_YEARS = set(range(2019, 2025))

assert set(home_ownership_raw["TIME_PERIOD"].unique()) == EXPECTED_YEARS, \
    "Unexpected temporal coverage in the home-ownership dataset."

assert home_ownership_raw["GEO"].nunique() == 27, \
    "Expected Switzerland plus the 26 Swiss cantons."

assert home_ownership_raw["OBS_VALUE"].isna().sum() == 0, \
    "Missing values detected in the target variable."

assert (home_ownership_raw["OBS_STATUS"] == "A").all(), \
    "Non-normal statistical observations detected."

print("Raw target validation passed.")

Raw target validation passed.


In [11]:
# Remove the national Swiss aggregate because the analytical unit
# of the ML dataset is canton-year, not country-year.
home_ownership = home_ownership_raw[
    home_ownership_raw["GEO"] != "CH"
].copy()

# Verify the resulting dimensions.
print(f"Shape after excluding Switzerland: {home_ownership.shape}")
print(f"Number of cantons: {home_ownership['GEO'].nunique()}")
print(f"Number of years: {home_ownership['TIME_PERIOD'].nunique()}")

Shape after excluding Switzerland: (156, 5)
Number of cantons: 26
Number of years: 6


In [12]:
# Keep only the variables required for the analytical dataset.
# OBS_STATUS is no longer needed because all observations were already
# validated as normal ("A") in the previous data-quality checks.
home_ownership = home_ownership[
    ["TIME_PERIOD", "GEO", "OBS_VALUE"]
].copy()

# Rename the variables using clear analytical names that will also
# be used later when integrating the different datasets.
home_ownership = home_ownership.rename(
    columns={
        "TIME_PERIOD": "YEAR",
        "OBS_VALUE": "HOME_OWNERSHIP_RATE",
    }
)

home_ownership.head()

,YEAR,GEO,HOME_OWNERSHIP_RATE
0,2019,CH040,27.7
1,2019,CH021,38.8
2,2019,CH061,34.4
3,2019,CH062,42.9
4,2019,CH063,39.4


In [13]:
# Validate the prepared target dataset before using it in later joins.
assert home_ownership.shape[0] == 156, \
    "Expected 156 canton-year observations."

assert home_ownership["GEO"].nunique() == 26, \
    "Expected 26 cantons."

assert home_ownership["YEAR"].nunique() == 6, \
    "Expected 6 years."

assert home_ownership.duplicated(["GEO", "YEAR"]).sum() == 0, \
    "Duplicate canton-year observations detected."

assert home_ownership["HOME_OWNERSHIP_RATE"].isna().sum() == 0, \
    "Missing values detected in the target."

print("Target preparation validation passed.")

Target preparation validation passed.


## 3. Household Features — Private Households by Household Size

This dataset contains the number of private households by household size for
Swiss cantons between 2019 and 2024.

The objective is to transform the original household-size categories into a small
set of interpretable features at canton-year level that can later be integrated
with the home-ownership target.

The raw dataset contains observations for Switzerland and the 26 cantons.
The national aggregate will be excluded to maintain the canton-year analytical unit.

In [14]:
# Load the household-size dataset through the reusable ingestion layer.
households_raw = load_dataset(HOUSEHOLDS_PATH)

# Display the first observations to verify that the dataset
# was parsed correctly.
households_raw.head()

,﻿STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,GEO_UNIT,"Cantons, districts and municipalities",HH_SIZE,Household size,FREQ,Frequency of observation,...,LAST_UPDATE,Update date,EMBARGO_DATE,Date of publication,DB_STATE,State of the database,OBS_STATUS,Observation Status,REGION_REF,Spatial reference
0,DATAFLOW,CH1.STATPOP:DF_STATPOP_PHH(1.0.0),Private households by institutional units and ...,I,LU,Lucerne,2,Two persons,A,Annual,...,2025-10-15T08:30,NaN,2025-10-15T08:30,NaN,2024-12-31,NaN,A,Normal value,POLG,Communes
1,DATAFLOW,CH1.STATPOP:DF_STATPOP_PHH(1.0.0),Private households by institutional units and ...,I,LU,Lucerne,2,Two persons,A,Annual,...,2025-10-15T08:30,NaN,2025-10-15T08:30,NaN,2024-12-31,NaN,A,Normal value,POLG,Communes
2,DATAFLOW,CH1.STATPOP:DF_STATPOP_PHH(1.0.0),Private households by institutional units and ...,I,LU,Lucerne,2,Two persons,A,Annual,...,2025-10-15T08:30,NaN,2025-10-15T08:30,NaN,2024-12-31,NaN,A,Normal value,POLG,Communes
3,DATAFLOW,CH1.STATPOP:DF_STATPOP_PHH(1.0.0),Private households by institutional units and ...,I,LU,Lucerne,2,Two persons,A,Annual,...,2025-10-15T08:30,NaN,2025-10-15T08:30,NaN,2024-12-31,NaN,A,Normal value,POLG,Communes
4,DATAFLOW,CH1.STATPOP:DF_STATPOP_PHH(1.0.0),Private households by institutional units and ...,I,LU,Lucerne,2,Two persons,A,Annual,...,2025-10-15T08:30,NaN,2025-10-15T08:30,NaN,2024-12-31,NaN,A,Normal value,POLG,Communes


In [15]:
# Inspect the structure of the raw household dataset before applying transformations.
# We verify its dimensions, temporal coverage, geographic coverage,
# household-size categories and observation status.

print(f"Shape: {households_raw.shape}")

print("\nYears:")
print(sorted(households_raw["TIME_PERIOD"].unique()))

print("\nNumber of geographies:")
print(households_raw["GEO_UNIT"].nunique())

print("\nHousehold-size categories:")
print(
    households_raw[
        ["HH_SIZE", "Household size"]
    ]
    .drop_duplicates()
    .sort_values("HH_SIZE")
)

print("\nObservation status:")
print(households_raw["OBS_STATUS"].value_counts(dropna=False))

print("\nMissing observation values:")
print(households_raw["OBS_VALUE"].isna().sum())

Shape: (1134, 26)

Years:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Number of geographies:
27

Household-size categories:
    HH_SIZE       Household size
66        1           One person
0         2          Two persons
6         3        Three persons
48        4         Four persons
144       5         Five persons
24       60  Six or more persons
12       _T                Total

Observation status:
OBS_STATUS
A    1134
Name: count, dtype: int64

Missing observation values:
0


In [16]:
# Remove the national Swiss aggregate because the analytical unit
# of the ML dataset is canton-year.
#
# In this dataset, Switzerland is represented by GEO_UNIT = "8100".
households = households_raw[
    households_raw["GEO_UNIT"] != "8100"
].copy()

# Verify the resulting dimensions and geographic coverage.
print(f"Shape after excluding Switzerland: {households.shape}")
print(f"Number of cantons: {households['GEO_UNIT'].nunique()}")
print(f"Number of years: {households['TIME_PERIOD'].nunique()}")
print(f"Number of household-size categories: {households['HH_SIZE'].nunique()}")

Shape after excluding Switzerland: (1092, 26)
Number of cantons: 26
Number of years: 6
Number of household-size categories: 7


In [17]:
# Keep only the variables required to construct the household features.
# The remaining columns contain metadata that is not needed for the
# analytical dataset.
households = households[
    [
        "TIME_PERIOD",
        "GEO_UNIT",
        "HH_SIZE",
        "Household size",
        "OBS_VALUE",
    ]
].copy()

# Rename the geographic and temporal variables using the same naming
# convention used in the target dataset.
households = households.rename(
    columns={
        "TIME_PERIOD": "YEAR",
        "GEO_UNIT": "GEO",
    }
)

households.head(10)

,YEAR,GEO,HH_SIZE,Household size,OBS_VALUE
0,2019,LU,2,Two persons,60968
1,2020,LU,2,Two persons,62330
2,2021,LU,2,Two persons,63205
3,2022,LU,2,Two persons,63855
4,2023,LU,2,Two persons,64851
5,2024,LU,2,Two persons,65902
6,2019,AG,3,Three persons,39073
7,2020,AG,3,Three persons,39312
8,2021,AG,3,Three persons,39383
9,2022,AG,3,Three persons,39648


In [18]:
# Remove the descriptive household-size label because HH_SIZE
# already identifies the same categories and is sufficient
# for the following transformations.
households = households[
    ["YEAR", "GEO", "HH_SIZE", "OBS_VALUE"]
].copy()

households.head()

,YEAR,GEO,HH_SIZE,OBS_VALUE
0,2019,LU,2,60968
1,2020,LU,2,62330
2,2021,LU,2,63205
3,2022,LU,2,63855
4,2023,LU,2,64851


In [19]:
# Check that each canton-year-household-size combination appears exactly once.
# This ensures that the dataset can be safely reshaped to one row per canton-year.
duplicate_households = households.duplicated(
    ["YEAR", "GEO", "HH_SIZE"]
).sum()

print(f"Duplicate canton-year-household-size observations: {duplicate_households}")

assert duplicate_households == 0, \
    "Duplicate canton-year-household-size observations detected."

Duplicate canton-year-household-size observations: 0


In [20]:
# Reshape the dataset from long to wide format.
# Each row now represents one canton-year observation,
# while each household-size category becomes a separate column.
households_wide = (
    households
    .pivot(
        index=["YEAR", "GEO"],
        columns="HH_SIZE",
        values="OBS_VALUE",
    )
    .reset_index()
)

# Remove the technical name assigned to the columns by the pivot.
households_wide.columns.name = None

print(f"Shape after pivot: {households_wide.shape}")

households_wide.head()

Shape after pivot: (156, 9)


,YEAR,GEO,1,2,3,4,5,60,_T
0,2019,AG,95954,104904,39073,39627,12868,4830,297256
1,2019,AI,2002,2165,796,856,512,177,6508
2,2019,AR,8173,8245,2916,2952,1207,515,24008
3,2019,BE,175404,165932,55207,53544,17878,6623,474588
4,2019,BL,42960,46310,16734,16408,4831,1501,128744


In [21]:
# Rename the household-size categories using descriptive names.
# This makes the following feature-engineering calculations easier to understand.
households_wide = households_wide.rename(
    columns={
        "1": "HH_1_PERSON",
        "2": "HH_2_PERSON",
        "3": "HH_3_PERSON",
        "4": "HH_4_PERSON",
        "5": "HH_5_PERSON",
        "60": "HH_6_PLUS_PERSON",
        "_T": "TOTAL_HOUSEHOLDS",
    }
)

households_wide.head()

,YEAR,GEO,HH_1_PERSON,HH_2_PERSON,HH_3_PERSON,HH_4_PERSON,HH_5_PERSON,HH_6_PLUS_PERSON,TOTAL_HOUSEHOLDS
0,2019,AG,95954,104904,39073,39627,12868,4830,297256
1,2019,AI,2002,2165,796,856,512,177,6508
2,2019,AR,8173,8245,2916,2952,1207,515,24008
3,2019,BE,175404,165932,55207,53544,17878,6623,474588
4,2019,BL,42960,46310,16734,16408,4831,1501,128744


In [22]:
# Calculate the total number of households from the six household-size categories.
# This allows us to validate the BFS total before using it as the denominator
# for the household-share features.
calculated_total = (
    households_wide["HH_1_PERSON"]
    + households_wide["HH_2_PERSON"]
    + households_wide["HH_3_PERSON"]
    + households_wide["HH_4_PERSON"]
    + households_wide["HH_5_PERSON"]
    + households_wide["HH_6_PLUS_PERSON"]
)

# Count observations where the calculated total differs from the
# total reported by BFS.
total_mismatches = (
    calculated_total != households_wide["TOTAL_HOUSEHOLDS"]
).sum()

print(f"Household total mismatches: {total_mismatches}")

assert total_mismatches == 0, \
    "Household-size categories do not sum to TOTAL_HOUSEHOLDS."

Household total mismatches: 0


In [23]:
# Create household-composition features while preserving the original
# household-size categories provided by BFS.
households_wide["SHARE_1_PERSON_HOUSEHOLDS"] = (
    households_wide["HH_1_PERSON"]
    / households_wide["TOTAL_HOUSEHOLDS"]
)

households_wide["SHARE_2_PERSON_HOUSEHOLDS"] = (
    households_wide["HH_2_PERSON"]
    / households_wide["TOTAL_HOUSEHOLDS"]
)

households_wide["SHARE_3_PERSON_HOUSEHOLDS"] = (
    households_wide["HH_3_PERSON"]
    / households_wide["TOTAL_HOUSEHOLDS"]
)

households_wide["SHARE_4_PERSON_HOUSEHOLDS"] = (
    households_wide["HH_4_PERSON"]
    / households_wide["TOTAL_HOUSEHOLDS"]
)

households_wide["SHARE_5_PERSON_HOUSEHOLDS"] = (
    households_wide["HH_5_PERSON"]
    / households_wide["TOTAL_HOUSEHOLDS"]
)

households_wide["SHARE_6_PLUS_PERSON_HOUSEHOLDS"] = (
    households_wide["HH_6_PLUS_PERSON"]
    / households_wide["TOTAL_HOUSEHOLDS"]
)

***REPORT NOTE:*** 
Household-size categories were initially preserved rather than manually aggregated in order to avoid introducing arbitrary grouping assumptions during data preparation. Feature selection or aggregation can subsequently be informed by EDA and modelling results.

In [24]:
# Verify that the household-size shares represent the complete household
# composition for each canton-year observation.
share_columns = [
    "SHARE_1_PERSON_HOUSEHOLDS",
    "SHARE_2_PERSON_HOUSEHOLDS",
    "SHARE_3_PERSON_HOUSEHOLDS",
    "SHARE_4_PERSON_HOUSEHOLDS",
    "SHARE_5_PERSON_HOUSEHOLDS",
    "SHARE_6_PLUS_PERSON_HOUSEHOLDS",
]

households_wide["SHARE_SUM"] = households_wide[share_columns].sum(axis=1)

# Display the range of the calculated sums.
print(f"Minimum share sum: {households_wide['SHARE_SUM'].min():.6f}")
print(f"Maximum share sum: {households_wide['SHARE_SUM'].max():.6f}")

# Use np.isclose because floating-point calculations may introduce
# very small numerical differences from exactly 1.0.
assert np.isclose(
    households_wide["SHARE_SUM"],
    1.0,
).all(), "Household shares do not sum to 1."

print("Household share validation passed.")

Minimum share sum: 1.000000
Maximum share sum: 1.000000
Household share validation passed.


In [25]:
# Keep only the household variables required for the analytical dataset.
# The original category counts and SHARE_SUM were intermediate variables
# used to construct and validate the household-composition features.
household_features = households_wide[
    [
        "YEAR",
        "GEO",
        "TOTAL_HOUSEHOLDS",
        "SHARE_1_PERSON_HOUSEHOLDS",
        "SHARE_2_PERSON_HOUSEHOLDS",
        "SHARE_3_PERSON_HOUSEHOLDS",
        "SHARE_4_PERSON_HOUSEHOLDS",
        "SHARE_5_PERSON_HOUSEHOLDS",
        "SHARE_6_PLUS_PERSON_HOUSEHOLDS",
    ]
].copy()

print(f"Final household feature shape: {household_features.shape}")

household_features.head()

Final household feature shape: (156, 9)


,YEAR,GEO,TOTAL_HOUSEHOLDS,SHARE_1_PERSON_HOUSEHOLDS,SHARE_2_PERSON_HOUSEHOLDS,SHARE_3_PERSON_HOUSEHOLDS,SHARE_4_PERSON_HOUSEHOLDS,SHARE_5_PERSON_HOUSEHOLDS,SHARE_6_PLUS_PERSON_HOUSEHOLDS
0,2019,AG,297256,0.322799,0.352908,0.131446,0.133309,0.043289,0.016249
1,2019,AI,6508,0.307621,0.332667,0.122311,0.131530,0.078672,0.027197
2,2019,AR,24008,0.340428,0.343427,0.121460,0.122959,0.050275,0.021451
3,2019,BE,474588,0.369592,0.349634,0.116326,0.112822,0.037671,0.013955
4,2019,BL,128744,0.333685,0.359706,0.129979,0.127447,0.037524,0.011659


In [26]:
# Validate the final household feature dataset before integration.
assert household_features.shape[0] == 156, \
    "Expected 156 canton-year observations."

assert household_features["GEO"].nunique() == 26, \
    "Expected 26 cantons."

assert household_features["YEAR"].nunique() == 6, \
    "Expected 6 years."

assert household_features.duplicated(["GEO", "YEAR"]).sum() == 0, \
    "Duplicate canton-year observations detected."

assert household_features.isna().sum().sum() == 0, \
    "Missing values detected in household features."

print("Household feature validation passed.")

Household feature validation passed.


***REPORT NOTE:*** 
__-Household Data Preparation-__ The original BFS household dataset was transformed from a long format, containing separate observations for each household-size category, into a canton-year analytical structure. After excluding the national Swiss aggregate, the dataset contained 156 unique canton-year observations covering all 26 cantons from 2019 to 2024. Household-size categories were converted into shares of total households to improve comparability across cantons of different sizes. The original household-size categories were preserved rather than manually aggregated in order to avoid introducing arbitrary grouping assumptions during data preparation. The resulting dataset was validated for completeness, uniqueness, missing values, and consistency of household totals and shares.

## 4. Demographic Features — Permanent Resident Population

This dataset provides demographic information on the permanent resident population
of Swiss cantons between 2019 and 2024.

The objective is to derive a small set of interpretable demographic features at
canton-year level for integration with the household and home-ownership datasets.

The dataset contains population breakdowns by sex, marital status and single-year
age groups. To avoid excessive dimensionality, individual age columns will not be
used directly as machine-learning features. Instead, they will later be aggregated
into broader demographic age groups.

In [27]:
# Load the demographic dataset through the reusable ingestion layer.
# This BFS CSV uses Latin-1 encoding, which is explicitly specified
# when loading the file.
population_raw = load_dataset(
    POPULATION_PATH,
    encoding="latin1",
)

print(f"Shape: {population_raw.shape}")

population_raw.head()

Shape: (3024, 107)


,Year,Canton,Population type,Sex,Marital status,Age - total,0 years,1 year,2 years,3 years,...,91 years,92 years,93 years,94 years,95 years,96 years,97 years,98 years,99 years,100 years or older
0,2019,Switzerland,Permanent resident population,Sex - total,Marital status - total,8606033,84500,87774,87829,89144,...,15694,12470,10071,7805,5934,4249,2860,2073,1232,1646
1,2019,Switzerland,Permanent resident population,Sex - total,Single,3843439,84500,87774,87829,89144,...,1016,856,734,567,427,351,209,170,120,147
2,2019,Switzerland,Permanent resident population,Sex - total,"Married, in a registered partnership",3617456,0,0,0,0,...,3791,2623,1861,1260,789,475,273,173,85,66
3,2019,Switzerland,Permanent resident population,Sex - total,"Widowed, partnership dissolved through death",405015,0,0,0,0,...,10004,8285,6970,5596,4405,3230,2238,1639,982,1345
4,2019,Switzerland,Permanent resident population,Sex - total,"Divorced, unmarried, partnership dissolved le...",739527,0,0,0,0,...,882,706,506,382,313,193,140,91,45,88


In [28]:
# Inspect the demographic dimensions before filtering.
# This confirms the exact category labels used by BFS and prevents
# filtering based on assumptions about the source data.

print("Population types:")
print(population_raw["Population type"].unique())

print("\nSex categories:")
print(population_raw["Sex"].unique())

print("\nMarital status categories:")
print(population_raw["Marital status"].unique())

print("\nYears:")
print(sorted(population_raw["Year"].unique()))

print("\nNumber of geographic values:")
print(population_raw["Canton"].nunique())

Population types:
<StringArray>
['Permanent resident population']
Length: 1, dtype: str

Sex categories:
<StringArray>
['Sex - total', 'Male', 'Female']
Length: 3, dtype: str

Marital status categories:
<StringArray>
[                             'Marital status - total',
                                              'Single',
                'Married, in a registered partnership',
        'Widowed, partnership dissolved through death',
 'Divorced, unmarried,  partnership dissolved legally',
                                       'No indication']
Length: 6, dtype: str

Years:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Number of geographic values:
28


In [29]:
# Keep only total permanent-resident-population observations.
# Sex and marital-status subtotals are excluded because the objective
# at this stage is to obtain one population observation per canton-year.
population = population_raw[
    (population_raw["Population type"] == "Permanent resident population")
    & (population_raw["Sex"] == "Sex - total")
    & (population_raw["Marital status"] == "Marital status - total")
].copy()

print(f"Shape after demographic filtering: {population.shape}")
print(f"Number of geographic values: {population['Canton'].nunique()}")
print(f"Number of years: {population['Year'].nunique()}")

population[
    ["Year", "Canton", "Age - total"]
].head(10)

Shape after demographic filtering: (168, 107)
Number of geographic values: 28
Number of years: 6


,Year,Canton,Age - total
0,2019,Switzerland,8606033
18,2019,Zürich,1539275
36,2019,Bern / Berne,1039426
54,2019,Luzern,413120
72,2019,Uri,36703
90,2019,Schwyz,160480
108,2019,Obwalden,37930
126,2019,Nidwalden,43087
144,2019,Glarus,40590
162,2019,Zug,127642


In [30]:
# Inspect all geographic values before excluding observations
# that do not correspond to Swiss cantons.
print(sorted(population["Canton"].unique()))

['Aargau', 'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'Basel-Landschaft', 'Basel-Stadt', 'Bern / Berne', 'Fribourg / Freiburg', 'Genève', 'Glarus', 'Graubünden / Grigioni / Grischun', 'Jura', 'Luzern', 'Neuchâtel', 'Nidwalden', 'No indication', 'Obwalden', 'Schaffhausen', 'Schwyz', 'Solothurn', 'St. Gallen', 'Switzerland', 'Thurgau', 'Ticino', 'Uri', 'Valais / Wallis', 'Vaud', 'Zug', 'Zürich']


In [31]:
# Keep only canton-level observations.
# "Switzerland" is the national aggregate, while "No indication"
# does not represent a specific canton.
population = population[
    ~population["Canton"].isin(["Switzerland", "No indication"])
].copy()

print(f"Shape after geographic filtering: {population.shape}")
print(f"Number of cantons: {population['Canton'].nunique()}")
print(f"Number of years: {population['Year'].nunique()}")

Shape after geographic filtering: (156, 107)
Number of cantons: 26
Number of years: 6


In [32]:
# Validate the expected canton-year structure before creating
# demographic features from the age columns.
assert population.shape[0] == 156
assert population["Canton"].nunique() == 26
assert population["Year"].nunique() == 6
assert population.duplicated(["Canton", "Year"]).sum() == 0

print("Population structure validation passed.")

Population structure validation passed.


In [33]:
# Define the age columns used to construct broader demographic groups.
# Aggregating single-year ages reduces dimensionality while preserving
# interpretable information about the population structure.

age_0_19 = [f"{age} years" for age in range(0, 20)]
age_20_39 = [f"{age} years" for age in range(20, 40)]
age_40_64 = [f"{age} years" for age in range(40, 65)]
age_65_plus = (
    [f"{age} years" for age in range(65, 100)]
    + ["100 years or older"]
)

# Verify that all expected age columns actually exist in the BFS dataset.
age_columns = age_0_19 + age_20_39 + age_40_64 + age_65_plus

missing_age_columns = [
    column for column in age_columns
    if column not in population.columns
]

print(f"Number of age columns: {len(age_columns)}")
print(f"Missing age columns: {missing_age_columns}")

Number of age columns: 101
Missing age columns: ['1 years']


___REPORT NOTE:___ The demographic dataset provides complete single-year age information from age 0 to 99, plus a 100-years-or-older category. These variables were preserved at the data-preparation stage, leaving age aggregation and feature engineering to the exploratory analysis stage.

In [34]:
population.head(10)

,Year,Canton,Population type,Sex,Marital status,Age - total,0 years,1 year,2 years,3 years,...,91 years,92 years,93 years,94 years,95 years,96 years,97 years,98 years,99 years,100 years or older
18,2019,Zürich,Permanent resident population,Sex - total,Marital status - total,1539275,16280,16642,16773,16822,...,2510,2075,1646,1266,963,661,438,335,204,245
36,2019,Bern / Berne,Permanent resident population,Sex - total,Marital status - total,1039426,9773,10131,10177,10296,...,2185,1772,1438,1081,802,595,375,279,152,220
54,2019,Luzern,Permanent resident population,Sex - total,Marital status - total,413120,4293,4412,4443,4352,...,733,535,406,321,243,178,103,80,42,79
72,2019,Uri,Permanent resident population,Sex - total,Marital status - total,36703,347,359,374,352,...,65,51,49,35,28,21,15,10,3,7
90,2019,Schwyz,Permanent resident population,Sex - total,Marital status - total,160480,1531,1661,1541,1564,...,265,189,144,104,72,59,29,25,22,26
108,2019,Obwalden,Permanent resident population,Sex - total,Marital status - total,37930,343,403,373,395,...,63,53,40,31,27,20,7,8,5,5
126,2019,Nidwalden,Permanent resident population,Sex - total,Marital status - total,43087,373,422,408,373,...,71,59,61,29,22,16,15,13,4,8
144,2019,Glarus,Permanent resident population,Sex - total,Marital status - total,40590,407,410,384,415,...,70,69,54,39,39,24,6,12,4,5
162,2019,Zug,Permanent resident population,Sex - total,Marital status - total,127642,1316,1351,1280,1395,...,204,149,101,88,61,48,18,16,5,14
180,2019,Fribourg / Freiburg,Permanent resident population,Sex - total,Marital status - total,321831,3470,3619,3515,3598,...,402,343,277,187,182,111,82,62,31,42


In [35]:
# Remove dimensions that became constant after filtering.
# They were necessary to select the correct BFS observations,
# but no longer provide information in the canton-year dataset.
population = population.drop(
    columns=[
        "Population type",
        "Sex",
        "Marital status",
    ]
)

# Standardize the identifier and total-population column names
# for later integration with the other project datasets.
population = population.rename(
    columns={
        "Year": "YEAR",
        "Canton": "CANTON",
        "Age - total": "TOTAL_POPULATION",
    }
)

print(f"Shape after removing constant dimensions: {population.shape}")
population.head()

Shape after removing constant dimensions: (156, 104)


,YEAR,CANTON,TOTAL_POPULATION,0 years,1 year,2 years,3 years,4 years,5 years,6 years,...,91 years,92 years,93 years,94 years,95 years,96 years,97 years,98 years,99 years,100 years or older
18,2019,Zürich,1539275,16280,16642,16773,16822,16584,16648,15816,...,2510,2075,1646,1266,963,661,438,335,204,245
36,2019,Bern / Berne,1039426,9773,10131,10177,10296,10316,10168,10003,...,2185,1772,1438,1081,802,595,375,279,152,220
54,2019,Luzern,413120,4293,4412,4443,4352,4283,4260,4223,...,733,535,406,321,243,178,103,80,42,79
72,2019,Uri,36703,347,359,374,352,344,343,343,...,65,51,49,35,28,21,15,10,3,7
90,2019,Schwyz,160480,1531,1661,1541,1564,1617,1511,1539,...,265,189,144,104,72,59,29,25,22,26


In [36]:
# Identify the 101 single-age columns exactly as they appear
# in the BFS dataset.
age_columns = (
    ["0 years", "1 year"]
    + [f"{age} years" for age in range(2, 100)]
    + ["100 years or older"]
)

# Verify that the complete age range is available.
assert len(age_columns) == 101
assert all(column in population.columns for column in age_columns)

print(f"Number of age columns: {len(age_columns)}")

Number of age columns: 101


In [37]:
# Validate that the sum of the single-age populations matches
# the total population reported by BFS for each canton-year.
calculated_population = population[age_columns].sum(axis=1)

population_mismatches = (
    calculated_population != population["TOTAL_POPULATION"]
).sum()

print(f"Population total mismatches: {population_mismatches}")

Population total mismatches: 0


## 5. Geographic Harmonisation

The three BFS datasets use different geographic identifiers for Swiss cantons.

- Home-ownership data uses geographic codes such as `CH040`.
- Household data uses canton abbreviations such as `ZH`.
- Population data uses canton names such as `Zürich`.

Before integrating the datasets, these identifiers must be mapped to a common
canton identifier. The project therefore uses the canton reference lookup to
harmonise geographic identifiers across all three sources.

In [38]:
# Load the canton reference table used to harmonise geographic
# identifiers across the three BFS datasets.
GEO_LOOKUP_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bfs"
    / "reference"
    / "GEO_CANTON_LOOKUP.xlsx"
)

geo_lookup = load_dataset(GEO_LOOKUP_PATH)

print(f"Shape: {geo_lookup.shape}")
print("\nColumns:")
print(geo_lookup.columns.tolist())

geo_lookup.head()

Shape: (27, 5)

Columns:
['CODE', 'LABEL_EN', 'LABEL_FR', 'LABEL_DE', 'LABEL_IT']


,CODE,LABEL_EN,LABEL_FR,LABEL_DE,LABEL_IT
0,CH,Switzerland,Suisse,Schweiz,Svizzera
1,CH011,Vaud,Vaud,Waadt,Vaud
2,CH012,Valais,Valais,Wallis,Vallese
3,CH013,Genève,Genève,Genf,Ginevra
4,CH021,Bern,Berne,Bern,Berna


In [39]:
# Remove the national aggregate from the geographic reference table.
# Only canton-level mappings are required for the analytical dataset.
canton_lookup = geo_lookup[
    geo_lookup["CODE"] != "CH"
].copy()

assert canton_lookup.shape[0] == 26

canton_lookup.head()

,CODE,LABEL_EN,LABEL_FR,LABEL_DE,LABEL_IT
1,CH011,Vaud,Vaud,Waadt,Vaud
2,CH012,Valais,Valais,Wallis,Vallese
3,CH013,Genève,Genève,Genf,Ginevra
4,CH021,Bern,Berne,Bern,Berna
5,CH022,Freiburg,Fribourg,Freiburg,Friburgo


In [40]:
# Check whether the population canton names match the English
# canton labels provided by the geographic reference table.
population_cantons = set(population["CANTON"])
lookup_cantons = set(canton_lookup["LABEL_EN"])

unmatched_population = sorted(
    population_cantons - lookup_cantons
)

print("Population canton names without a direct lookup match:")
print(unmatched_population)

Population canton names without a direct lookup match:
['Bern / Berne', 'Fribourg / Freiburg', 'Graubünden / Grigioni / Grischun', 'Valais / Wallis']


In [41]:
# Normalize the four multilingual canton labels used differently
# in the population dataset and the geographic reference table.
population_name_mapping = {
    "Bern / Berne": "Bern",
    "Fribourg / Freiburg": "Freiburg",
    "Graubünden / Grigioni / Grischun": "Graubünden",
    "Valais / Wallis": "Valais",
}

population["CANTON_LOOKUP_NAME"] = (
    population["CANTON"]
    .replace(population_name_mapping)
)

# Verify that all 26 population canton names can now be found
# in the official geographic lookup.
unmatched_population = set(population["CANTON_LOOKUP_NAME"]) - set(
    canton_lookup["LABEL_EN"]
)

assert len(unmatched_population) == 0

print("All population cantons matched successfully.")

All population cantons matched successfully.


In [42]:
# Add the standardized BFS geographic code to the population dataset.
# This creates the common geographic identifier that will later be
# used to integrate the three datasets.
population = population.merge(
    canton_lookup[
        ["CODE", "LABEL_EN"]
    ],
    left_on="CANTON_LOOKUP_NAME",
    right_on="LABEL_EN",
    how="left",
    validate="many_to_one",
)

# Rename the standardized geographic code consistently with the target dataset.
population = population.rename(
    columns={
        "CODE": "GEO",
    }
)

# Remove temporary columns used only for geographic harmonisation.
population = population.drop(
    columns=[
        "CANTON_LOOKUP_NAME",
        "LABEL_EN",
    ]
)

print(f"Shape: {population.shape}")
print(f"Missing GEO codes: {population['GEO'].isna().sum()}")
print(f"Unique GEO codes: {population['GEO'].nunique()}")

population[
    ["YEAR", "CANTON", "GEO", "TOTAL_POPULATION"]
].head()

Shape: (156, 105)
Missing GEO codes: 0
Unique GEO codes: 26


,YEAR,CANTON,GEO,TOTAL_POPULATION
0,2019,Zürich,CH040,1539275
1,2019,Bern / Berne,CH021,1039426
2,2019,Luzern,CH061,413120
3,2019,Uri,CH062,36703
4,2019,Schwyz,CH063,160480


In [43]:
# Inspect the canton abbreviations used by the household dataset
# before mapping them to the standardized BFS geographic codes.
household_cantons = sorted(
    household_features["GEO"].unique()
)

print(f"Number of household canton codes: {len(household_cantons)}")
print(household_cantons)

Number of household canton codes: 26
['AG', 'AI', 'AR', 'BE', 'BL', 'BS', 'FR', 'GE', 'GL', 'GR', 'JU', 'LU', 'NE', 'NW', 'OW', 'SG', 'SH', 'SO', 'SZ', 'TG', 'TI', 'UR', 'VD', 'VS', 'ZG', 'ZH']


In [44]:
# Map the canton abbreviations used in the household dataset
# to the standardized BFS geographic codes used by the target dataset.
canton_abbreviation_to_geo = {
    "VD": "CH011",
    "VS": "CH012",
    "GE": "CH013",
    "BE": "CH021",
    "FR": "CH022",
    "SO": "CH023",
    "BS": "CH024",
    "BL": "CH025",
    "AG": "CH031",
    "ZH": "CH040",
    "GL": "CH051",
    "SH": "CH052",
    "AR": "CH053",
    "AI": "CH054",
    "SG": "CH055",
    "GR": "CH056",
    "TG": "CH057",
    "LU": "CH061",
    "UR": "CH062",
    "SZ": "CH063",
    "OW": "CH064",
    "NW": "CH065",
    "ZG": "CH066",
    "TI": "CH070",
    "NE": "CH032",
    "JU": "CH033",
}

# Verify that the mapping covers exactly the 26 canton
# abbreviations present in the household dataset.
unmapped_households = set(household_features["GEO"]) - set(
    canton_abbreviation_to_geo
)

assert len(unmapped_households) == 0
assert len(canton_abbreviation_to_geo) == 26

print("All household canton abbreviations are covered.")

All household canton abbreviations are covered.


In [45]:
# Replace canton abbreviations with the standardized BFS geographic codes.
household_features["GEO"] = household_features["GEO"].map(
    canton_abbreviation_to_geo
)

# Validate the geographic harmonisation.
assert household_features["GEO"].isna().sum() == 0
assert household_features["GEO"].nunique() == 26

# Verify that households and the target contain exactly
# the same set of canton codes.
assert set(household_features["GEO"]) == set(home_ownership["GEO"])

print("Household geographic harmonisation passed.")

household_features[
    ["YEAR", "GEO", "TOTAL_HOUSEHOLDS"]
].head()

Household geographic harmonisation passed.


,YEAR,GEO,TOTAL_HOUSEHOLDS
0,2019,CH031,297256
1,2019,CH054,6508
2,2019,CH053,24008
3,2019,CH021,474588
4,2019,CH025,128744


In [46]:
# Merge the home-ownership target with the household features.
# The one-to-one validation ensures that each canton-year appears
# exactly once in both datasets.
analytical_dataset = home_ownership.merge(
    household_features,
    on=["YEAR", "GEO"],
    how="inner",
    validate="one_to_one",
)

print(f"Shape after target + households: {analytical_dataset.shape}")

Shape after target + households: (156, 10)


In [47]:
# Add the demographic data using the same canton-year analytical key.
analytical_dataset = analytical_dataset.merge(
    population,
    on=["YEAR", "GEO"],
    how="inner",
    validate="one_to_one",
)

print(f"Final analytical dataset shape: {analytical_dataset.shape}")
print(f"Number of cantons: {analytical_dataset['GEO'].nunique()}")
print(f"Number of years: {analytical_dataset['YEAR'].nunique()}")

Final analytical dataset shape: (156, 113)
Number of cantons: 26
Number of years: 6


In [48]:
# Final validation of the integrated analytical dataset.
assert analytical_dataset.shape[0] == 156
assert analytical_dataset["GEO"].nunique() == 26
assert analytical_dataset["YEAR"].nunique() == 6
assert analytical_dataset.duplicated(["GEO", "YEAR"]).sum() == 0
assert analytical_dataset["HOME_OWNERSHIP_RATE"].isna().sum() == 0
assert analytical_dataset.isna().sum().sum() == 0

print("Final analytical dataset validation passed.")

Final analytical dataset validation passed.


**Report Note:**
<br>
*Analytical Dataset Construction:* The three selected BFS data sources were harmonised at canton-year level using standardized geographic identifiers. Differences in geographic representation across the sources—including BFS geographic codes, canton abbreviations and multilingual canton names—were resolved before integration. The datasets were subsequently joined using the combination of year and canton as a unique key. The resulting analytical dataset contains 156 observations, representing all 26 Swiss cantons over the 2019–2024 period, with no missing canton-year observations introduced during integration.

In [49]:
# Define the directory for processed analytical datasets.
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

# Define the output path for the integrated canton-year dataset.
ANALYTICAL_DATA_PATH = (
    PROCESSED_DATA
    / "canton_year_analytical_dataset_2019_2024.csv"
)

# Export the analytical dataset without the pandas index.
analytical_dataset.to_csv(
    ANALYTICAL_DATA_PATH,
    index=False,
)

print(f"Analytical dataset exported to: {ANALYTICAL_DATA_PATH}")

Analytical dataset exported to: G:\Otros ordenadores\Mi MacBook Pro\HSLU-CAS_ML-CAS Project\swiss-data-intelligence\data\processed\canton_year_analytical_dataset_2019_2024.csv


In [50]:
# Inspect the canton reference table used for geographic harmonisation.
display(
    canton_lookup[
        [
            "CODE",
            "LABEL_EN",
            "LABEL_DE",
        ]
    ]
    .sort_values("CODE")
    .reset_index(drop=True)
)

,CODE,LABEL_EN,LABEL_DE
0,CH011,Vaud,Waadt
1,CH012,Valais,Wallis
2,CH013,Genève,Genf
3,CH021,Bern,Bern
4,CH022,Freiburg,Freiburg
5,CH023,Solothurn,Solothurn
6,CH024,Neuchâtel,Neuenburg
7,CH025,Jura,Jura
8,CH031,Basel-Stadt,Basel-Stadt
9,CH032,Basel-Landschaft,Basel-Landschaft
